# Vérification des versions Python

In [ ]:
import sys

print(f"Python executable: {sys.executable}")
print(f"Python version: {sys.version}")

Python executable: c:\Users\Administrateur\Documents\M2i\.venv\Scripts\python.exe
Python version: c:\Users\Administrateur\Documents\M2i\.venv\Scripts\python.exe


## Vérification des versions pandas

In [2]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

print(f"Pandas version: {pd.__version__}")

Pandas version: 3.0.5


### Récupération du fichier/semaine

In [9]:
import os
from datetime import datetime

import pycurl


def progress(download_t, download_d, upload_t, upload_d):
    download_done = 0
    if download_t > 0:
        download_done = download_d / download_t
    print(f"               \r{download_done}", end='')

today_calendar = datetime.now().isocalendar()

off_parquet_url = "https://huggingface.co/datasets/openfoodfacts/product-database/resolve/main/food.parquet?download=true"
# food_<year>_<week num>.parquet
off_parquet_path = f"../data/food_{today_calendar[0]}_{today_calendar[1]}.parquet"

if not os.path.exists(off_parquet_path):
    with open(off_parquet_path, "wb") as cache_file:
        curl = pycurl.Curl()
        curl.setopt(curl.URL, off_parquet_url)
        curl.setopt(curl.WRITEDATA, cache_file)
        curl.setopt(curl.NOPROGRESS, False)
        curl.setopt(curl.XFERINFOFUNCTION, progress)
        curl.setopt(curl.FOLLOWLOCATION, 1)
        curl.perform()
        curl.close()

else:
    print("Fichier parquet déjà présent...")

1.0                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     

### Inventaire des colonnes

In [ ]:
parquet_file = pq.ParquetFile(off_parquet_path)
very_light_batch = next(parquet_file.iter_batches(1))
very_light_df = very_light_batch.to_pandas()


3. Inventaire des colonnes

In [4]:
off_code_df = pd.read_parquet(off_parquet_path, columns=["code"])

total_product = off_code_df.shape[0]
print(f"Nombre de colonnes: {len(very_light_df.columns)}")
print(f"Nombre de produits: {total_product}")

print("| Colonne | Type | % Remplissage | | Mémoire (MB) |")
print("| --- | --- | --- | --- |")
for col in very_light_df.columns:
    off_col_df = pd.read_parquet(off_parquet_path, columns=[col])

    mask = off_col_df[col].notna()
    memory = off_col_df.memory_usage(deep=True).sum() / 1024**2

    print(f"| {col} | {very_light_df[col].dtype} | {round(mask.sum() / total_product * 100.0, 2)} | {round(memory, 2)} |")


Nombre de colonnes: 111
Nombre de produits: 4636471
| Colonne | Type | % Remplissage | | Mémoire (MB) |
| --- | --- | --- | --- |
| additives_n | int32 | 27.96 | 35.37 |
| additives_tags | object | 29.0 | 229.21 |
| allergens_tags | object | 99.07 | 526.66 |
| brands_tags | object | 65.61 | 384.61 |
| brands | str | 65.6 | 66.09 |
| categories | str | 40.06 | 171.23 |
| categories_tags | object | 43.59 | 291.17 |
| categories_properties | object | 99.06 | 842.01 |
| checkers_tags | object | 100.0 | 530.6 |
| ciqual_food_name_tags | object | 29.75 | 232.38 |
| cities_tags | object | 20.71 | 194.04 |
| code | str | 100.0 | 92.14 |
| compared_to_category | str | 38.54 | 63.32 |
| complete | int32 | 100.0 | 35.37 |
| completeness | float32 | 100.0 | 17.69 |
| correctors_tags | object | 100.0 | 530.6 |
| countries_tags | object | 99.79 | 529.71 |
| created_t | int64 | 100.0 | 35.37 |
| creator | str | 100.0 | 80.48 |
| data_quality_errors_tags | object | 99.07 | 526.66 |
| data_quality_info

4. Cinq questions à résoudre en équipe

- combien de produits vendus en France ? 

In [17]:
import json


def filter_by_country(value, country):
    try:
        # Transforme la chaîne JSON en véritable liste Python
        list_array = list(value)
        return country in list_array
    except (json.JSONDecodeError, TypeError):
        # Gestion des erreurs si le JSON est malformé ou s'il y a un NaN
        return False

off_country_df = pd.read_parquet(off_parquet_path, columns=["countries_tags"])
mask = off_country_df["countries_tags"].apply(lambda x: filter_by_country(x, "en:france"))

nb_value_france = mask.sum()
print(f"Nombre de produits total : {len(off_country_df):,}")
print(f"Produits vendus en France : {nb_value_france:,}")

Nombre de produits total : 4,636,471
Produits vendus en France : 1,247,336


- quelle part a un Nutri-Score renseigné ?

In [16]:
off_nutriscore_grade_df = pd.read_parquet(off_parquet_path, columns=["nutriscore_grade"])

mask = (
        (off_nutriscore_grade_df["nutriscore_grade"].notna())
        & (off_nutriscore_grade_df["nutriscore_grade"] != "not-applicable")
        & (off_nutriscore_grade_df["nutriscore_grade"] != "unknown")
)

total_product = len(off_nutriscore_grade_df)
print(f"Part des produits avec un nutriscore renseigés: {round(mask.sum() / total_product * 100.0, 2)}")


Part des produits avec un nutriscore renseigés: 29.79


- les dix marques les plus présentes ?

In [14]:
off_brands_df = pd.read_parquet(off_parquet_path, columns=["brands"])
print(off_brands_df[off_brands_df["brands"] != ""].value_counts().head(10))

brands   
Carrefour    20732
Coop         14545
Lidl         14243
U            12384
Aldi         12353
BonÀrea      12155
Hacendado    10658
Auchan       10590
Tesco        10503
Delhaize      9918
Name: count, dtype: int64


- le taux de manquants sur les nutriments clés ( energy_100g , sugars_100g , salt_100g ) ?

In [ ]:
nutriments_list = ["energy", "sugars", "salt"]

def extraire_depuis_liste(liste_nutriments, nom_nutriment, extract_value : bool):
    if not isinstance(liste_nutriments, (np.ndarray, list)):
        return None

    for nutriment in liste_nutriments:
        if isinstance(nutriment, dict) and nutriment.get('name') == nom_nutriment:

            if extract_value:
                return nutriment.get('100g')
            
            return nutriment.get('unit')
        
    return None

off_nutriments_df = pd.read_parquet(off_parquet_path, columns=["nutriments"])

# Application sur le DataFrame Pandas
for nutriment in nutriments_list:
    off_nutriments_df[nutriment + "_100g"] = off_nutriments_df['nutriments'].apply(lambda x: extraire_depuis_liste(x, nutriment, True))
    off_nutriments_df[nutriment + "_unit"] = off_nutriments_df['nutriments'].apply(lambda x: extraire_depuis_liste(x, nutriment, False))


Taux de manquant pour le energy: 27.69
Taux de manquant pour le sugars: 32.44
Taux de manquant pour le salt: 37.44


In [13]:

total_product = len(off_nutriments_df)
for nutriment in nutriments_list:
    mask = off_nutriments_df[nutriment + "_100g"].isna()
    print(f"Taux de manquant pour {nutriment}: {round(mask.sum() / total_product * 100.0, 2)}")
    


Taux de manquant pour energy: 27.69
Taux de manquant pour sugars: 32.44
Taux de manquant pour salt: 37.44


- qu'est-ce qui vous semble le plus « sale » dans ces données ?

In [11]:
for nutriment in nutriments_list:
    print(off_nutriments_df[nutriment + "_unit"].value_counts())


energy_unit
kJ              2424359
kcal             943846
kj                 1144
Cal                   6
กิโลแคลอรี            6
Kcal                  4
kilocalories          2
cal                   2
                      1
KJ                    1
calories              1
Calories              1
mg                    1
kilocalorie           1
g                     1
กิโลแคลอรี่           1
kk                    1
The kcal              1
kca                   1
Name: count, dtype: int64
sugars_unit
g       3141199
            921
mg           23
G             9
0             6
1             3
16g           2
µg            2
2             1
ก.            1
10            1
10.4          1
l11g          1
3g            1
9             1
1g            1
Name: count, dtype: int64
salt_unit
g          2622728
mg          288765
               831
µg              31
Mg               8
0                3
&#181;g          2
0\nmg            1
5mg              1
0.028            1
1800 mg     

La disparitée des unitées